<a href="https://colab.research.google.com/github/akashde1998-Alpha/GCN-by-pytorch-/blob/main/GAT_in_pytorch_geometric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This cell installs the necessary Python packages: `torch` and `torch_geometric`.

In [51]:
!pip install torch
!pip install torch_geometric

This cell imports the required libraries for building and training a Graph Neural Network, including `torch`, `torch.nn.functional` (as `F`), `torch_geometric`, `NormalizeFeatures` from `torch_geometric.transforms`, and `Planetoid` from `torch_geometric.datasets`.

In [70]:
import torch
import torch.nn.functional as F

import torch_geometric
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Planetoid

This cell loads the 'Cora' dataset from the `Planetoid` collection. It applies `NormalizeFeatures()` to normalize node features and stores the data in the `data` variable.

In [72]:
dataset=torch_geometric.datasets.Planetoid(root='/Akash', name='Cora', transform=NormalizeFeatures())
data=dataset[0]

This cell defines the Graph Attention Network (GAT) model using `torch_geometric.nn.GATConv`. It's a two-layer GAT with dropout and ELU activation. An instance of this GAT model is then created.

In [73]:
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):

  def __init__(self,hidden_channels,heads):
   super().__init__()
   torch.manual_seed(123456)

   self.conv1=GATConv(dataset.num_features, hidden_channels, heads=heads,dropout=0.6, concat=True)
   self.conv2=GATConv(hidden_channels*heads, dataset.num_classes, heads=1,dropout=0.6, concat=False)

  def forward(self, x, edge_index):
    x = F.dropout(x, p=0.6, training=self.training)
    x=self.conv1(x,edge_index)
    x=F.elu(x)
    x=F.dropout(x,p=0.6, training=self.training)
    x=self.conv2(x,edge_index)
    return x
model=GAT(hidden_channels=8, heads=8)
print(model)

GAT(
  (conv1): GATConv(1433, 8, heads=8)
  (conv2): GATConv(64, 7, heads=1)
)


This cell initializes the `Adam` optimizer with a specified learning rate and weight decay, and sets up the `CrossEntropyLoss` as the criterion for the model's training process.

In [74]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
Criterion=torch.nn.CrossEntropyLoss()

This cell defines the `train` function, which orchestrates a single training step. It performs a forward pass, calculates the loss, backpropagates gradients, and updates the model's parameters.

In [75]:
def train():
 model.train()
 optimizer.zero_grad()
 out=model(data.x, data.edge_index)
 loss=Criterion(out[data.train_mask], data.y[data.train_mask])
 loss.backward()
 optimizer.step()
 return loss



This cell defines the `test` function, which evaluates the model's performance. It sets the model to evaluation mode, performs a forward pass, predicts class labels, and calculates the accuracy on the test set.

In [76]:
def test():
  model.eval()
  out=model(data.x, data.edge_index)
  pred=out.argmax(dim=1)
  test_correct=(pred[data.test_mask]==data.y[data.test_mask])
  test_acc=(int(test_correct.sum())/int(data.test_mask.sum()))
  return test_acc


This cell executes the training loop for 200 epochs, calling the `train()` function in each epoch and printing the epoch number and the corresponding loss.

In [77]:
for epoch in range(1, 201):

    loss = train()

    print(f'Epoch: {epoch:03d}, ' f'Loss: {loss:.4f}')

Epoch: 001, Loss: 1.9478
Epoch: 002, Loss: 1.9401
Epoch: 003, Loss: 1.9358
Epoch: 004, Loss: 1.9269
Epoch: 005, Loss: 1.9214
Epoch: 006, Loss: 1.9134
Epoch: 007, Loss: 1.8963
Epoch: 008, Loss: 1.9023
Epoch: 009, Loss: 1.8919
Epoch: 010, Loss: 1.8772
Epoch: 011, Loss: 1.8682
Epoch: 012, Loss: 1.8780
Epoch: 013, Loss: 1.8664
Epoch: 014, Loss: 1.8487
Epoch: 015, Loss: 1.8413
Epoch: 016, Loss: 1.8156
Epoch: 017, Loss: 1.8454
Epoch: 018, Loss: 1.8152
Epoch: 019, Loss: 1.7808
Epoch: 020, Loss: 1.7988
Epoch: 021, Loss: 1.7820
Epoch: 022, Loss: 1.7840
Epoch: 023, Loss: 1.7600
Epoch: 024, Loss: 1.7518
Epoch: 025, Loss: 1.7334
Epoch: 026, Loss: 1.7164
Epoch: 027, Loss: 1.7196
Epoch: 028, Loss: 1.6977
Epoch: 029, Loss: 1.7065
Epoch: 030, Loss: 1.6698
Epoch: 031, Loss: 1.6411
Epoch: 032, Loss: 1.6521
Epoch: 033, Loss: 1.6549
Epoch: 034, Loss: 1.6043
Epoch: 035, Loss: 1.5968
Epoch: 036, Loss: 1.5998
Epoch: 037, Loss: 1.5538
Epoch: 038, Loss: 1.5608
Epoch: 039, Loss: 1.5494
Epoch: 040, Loss: 1.5693


This cell calculates the final test accuracy after the training loop has completed by calling the `test()` function and then prints the result.

In [78]:

test_acc = test()
print("test_acc:", test_acc)

test_acc: 0.823
